In [7]:
import pandas as pd
import numpy as np
import math

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [8]:
FILE_PATH = r"C:\Users\felip\Documents\football-data-co-uk\data\raw\brazil_campeonato_brasileiro\BRA.csv"

df = pd.read_csv(FILE_PATH, encoding="utf-8-sig", parse_dates=["Date"], dayfirst=True,).sort_values("Date")
df = df[~(df["HG"].isna() & df["AG"].isna())]   # Remove the matches that have no score (HG and AG are NaN) 2016-12-11 19:00 Chapecoense-SC	x Atletico-MG
df = df.drop(columns=["PSCH", "PSCD", "PSCA"])  # Remove the columns (PSCH, PSCD, PSCA)
df = df.convert_dtypes({"Date": "datetime64[us]", "HG": "int64", "AG": "int64"})

print(df.shape, df.dtypes)

(5518, 22) Country            string
League             string
Season              Int64
Date       datetime64[us]
Time               string
Home               string
Away               string
HG                  Int64
AG                  Int64
Res                string
MaxCH             Float64
MaxCD             Float64
MaxCA             Float64
AvgCH             Float64
AvgCD             Float64
AvgCA             Float64
BFECH             Float64
BFECD             Float64
BFECA             Float64
B365CH            Float64
B365CD            Float64
B365CA            Float64
dtype: object


In [9]:
df.tail()

,Country,League,Season,Date,Time,Home,Away,HG,AG,Res,MaxCH,MaxCD,MaxCA,AvgCH,AvgCD,AvgCA,BFECH,BFECD,BFECA,B365CH,B365CD,B365CA
5517,Brazil,Serie A,2026,2026-07-26,23:30,Palmeiras,Atletico-MG,1,2,A,1.62,3.85,6.25,1.57,3.71,5.78,1.68,3.85,6.8,1.57,3.6,6.25
5512,Brazil,Serie A,2026,2026-07-26,20:00,Bahia,Corinthians,1,1,D,2.4,3.2,3.25,2.3,3.13,3.05,2.46,3.3,3.35,2.4,3.2,3.1
5511,Brazil,Serie A,2026,2026-07-26,00:30,Vasco,Mirassol,1,1,D,2.05,3.4,3.9,1.98,3.28,3.68,2.08,3.6,4.1,1.96,3.4,3.9
5513,Brazil,Serie A,2026,2026-07-26,20:00,Cruzeiro,Botafogo RJ,0,1,A,1.78,3.9,4.6,1.72,3.77,4.27,1.81,4.0,4.8,1.68,3.9,4.33
5518,Brazil,Serie A,2026,2026-07-26,23:30,Remo,Vitoria,2,0,H,2.46,3.35,3.2,2.31,3.23,2.92,2.5,3.4,3.2,2.35,3.3,2.75


#### Feature Enginering

In [10]:
df["TG"] = df["HG"] + df["AG"]
df["0.5-"] = (df["TG"] < 0.5).astype(int)
df["0.5+"] = (df["TG"] > 0.5).astype(int)
df["1.5+"] = (df["TG"] > 1.5).astype(int)
df["2.5+"] = (df["TG"] > 2.5).astype(int)
df["3.5+"] = (df["TG"] > 3.5).astype(int)
df["4.5+"] = (df["TG"] > 4.5).astype(int)
df["5.5+"] = (df["TG"] > 5.5).astype(int)
df["BTTS"] = ((df["HG"] > 0) & (df["AG"] > 0)).astype(int)
df["over_1.5+_and_btts"] = ((df["1.5+"] == 1) & (df["BTTS"] == 1)).astype(int)
df["over_2.5+_and_btts"] = ((df["2.5+"] == 1) & (df["BTTS"] == 1)).astype(int)
df["over_1.5+_and_btts"] = ((df["1.5+"] == 1) & (df["BTTS"] == 1)).astype(int)
df["over_3.5+_and_btts"] = ((df["3.5+"] == 1) & (df["BTTS"] == 1)).astype(int)
df["score_difference"] = df["HG"] - df["AG"]
df["absolute_score_difference"] = df["score_difference"].abs()

# Home team features
df["home_points"] = df["Res"].map({"H": 3, "D": 1, "A": 0})
df["home_won"] = (df["Res"] == "H").astype(int)  # home team(H) won the match
df["home_drew"] = (df["Res"] == "D").astype(int) # home team(H) drew the match
df["home_lost"] = (df["Res"] == "A").astype(int) # home team(H) lost the match
df["home_won_or_drew"] = df["Res"].isin(["H","D"]).astype(int)  # home team(H) won or drew the match
df["home_team_0.5-"] = (df["HG"] < 0.5).astype(int) # home team(H) < 0.5
df["home_team_0.5+"] = (df["HG"] > 0.5).astype(int) # home team(H) > 0.5
df["home_team_1.5+"] = (df["HG"] > 1.5).astype(int) # home team(H) > 1.5
df["home_team_2.5+"] = (df["HG"] > 2.5).astype(int) # home team(H) > 2.5
df["home_team_3.5+"] = (df["HG"] > 3.5).astype(int) # home team(H) > 3.5
df["home_team_4.5+"] = (df["HG"] > 4.5).astype(int) # home team(H) > 4.5
df["home_clean_sheet"] = (df["AG"] == 0).astype(int) # home team(H) kept a clean sheet
df["home_or_draw_and_over_1.5"] = ((df["home_won_or_drew"] == 1) & (df["home_team_1.5+"] == 1)).astype(int)
df["home_win_clean_sheet"] = ((df["home_won"] == 1) & (df["home_clean_sheet"] == 1)).astype(int)
df["home_goals_sum_last_5"] = (df.groupby("Home")["HG"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum()).astype("Int32"))
df["home_goals_mean_last_5"] = (df.groupby("Home")["HG"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))
df["home_goals_std_last_5"] = (df.groupby("Home")["HG"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).std()))
df["home_conceded_sum_last_5"] = (df.groupby("Home")["AG"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum()).astype("Int32")) # Goals conceded in the last 5 matches
df["home_dominance"] = ((df["HG"] + 1) / (df["AG"] + 1)).round(2) # Goal Dominance Index
df["home_points_last_5"] = df.groupby("Home")["home_points"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum().astype("Int32")) # Rolling points in last 5 matches
df["home_win_streak"] = df.groupby("Home")["home_won"].transform(lambda x: x.shift(1)).fillna(0).astype("Int32")
df["home_cumulative_points_season"] = df.groupby(["Season", "Home"])["home_points"].transform(lambda x: x.shift(1).cumsum())


# Away team features
df["away_points"] = df["Res"].map({"A": 3, "D": 1, "H": 0})
df["away_won"] = (df["Res"] == "A").astype(int)  # away team(A) won the match
df["away_drew"] = (df["Res"] == "D").astype(int) # away team(A) drew the match
df["away_lost"] = (df["Res"] == "H").astype(int) # away team(A) lost the match
df["away_won_or_drew"] = df["Res"].isin(["A","D"]).astype(int)  # away team(A) won or drew the match
df["away_team_0.5-"] = (df["AG"] < 0.5).astype(int) # away team(A) < 0.5
df["away_team_0.5+"] = (df["AG"] > 0.5).astype(int) # away team(A) > 0.5
df["away_team_1.5+"] = (df["AG"] > 1.5).astype(int) # away team(A) > 1.5
df["away_team_2.5+"] = (df["AG"] > 2.5).astype(int) # away team(A) > 2.5
df["away_team_3.5+"] = (df["AG"] > 3.5).astype(int) # away team(A) > 3.5
df["away_team_4.5+"] = (df["AG"] > 4.5).astype(int) # away team(A) > 4.5
df["away_clean_sheet"] = (df["HG"] == 0).astype(int) # away team(A) kept a clean sheet
df["away_or_draw_and_over_1.5"] = ((df["away_won_or_drew"] == 1) & (df["away_team_1.5+"] == 1)).astype(int)
df["away_win_clean_sheet"] = ((df["away_won"] == 1) & (df["away_clean_sheet"] == 1)).astype(int)
df["away_goals_sum_last_5"] = (df.groupby("Away")["AG"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum().astype("Int32")))
df["away_goals_mean_last_5"] = (df.groupby("Away")["AG"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))
df["away_goals_std_last_5"] = (df.groupby("Away")["AG"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).std()))
df["away_conceded_sum_last_5"] = (df.groupby("Away")["HG"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum()).astype("Int32"))
df["away_dominance"] = ((df["AG"] + 1) / (df["HG"] + 1)).round(2) # Goal Dominance Index
df["away_points_last_5"] = df.groupby("Away")["away_points"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum().astype("Int32")) # Rolling points in last 5 matches
df["away_win_streak"] = df.groupby("Away")["away_won"].transform(lambda x: x.shift(1)).fillna(0).astype("Int32")
df["away_cumulative_points_season"] = df.groupby(["Season", "Away"])["away_points"].transform(lambda x: x.shift(1).cumsum())

# Home and Away Indicators
df["attack_vs_defense_home"] = (df["home_goals_mean_last_5"] - (df["away_conceded_sum_last_5"] / 5)).round(2)
df["attack_vs_defense_away"] = (df["away_goals_mean_last_5"] - (df["home_conceded_sum_last_5"] / 5)).round(2)
df["form_differential_last_5"] = df["home_points_last_5"] - df["away_points_last_5"] # Form Differential (Home form - Away form)
df["h2h_key"] = df.apply(lambda x: tuple(sorted([x["Home"], x["Away"]])), axis=1) # Head-to-head history
df["h2h_home_win_rate_last_5"] = (df.sort_values("Date").groupby("h2h_key")["home_won"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))

# Temporal features
df["day_of_week"] = df["Date"].dt.dayofweek  # 0=Monday, 6=Sunday
df["month"] = df["Date"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["home_rest_days"] = df.groupby("Home")["Date"].diff().dt.days # Days of home rest since last match per team
df["away_rest_days"] = df.groupby("Away")["Date"].diff().dt.days # Days of away rest since last match per team
df["rest_advantage"] = df["home_rest_days"] - df["away_rest_days"] # Rest advantage (home rest days - away rest days)

df.tail(30)

,Country,League,Season,Date,Time,Home,Away,HG,AG,Res,MaxCH,MaxCD,MaxCA,AvgCH,AvgCD,AvgCA,BFECH,BFECD,BFECA,B365CH,B365CD,B365CA,TG,0.5-,0.5+,1.5+,2.5+,3.5+,4.5+,5.5+,BTTS,over_1.5+_and_btts,over_2.5+_and_btts,over_3.5+_and_btts,score_difference,absolute_score_difference,home_points,home_won,home_drew,home_lost,home_won_or_drew,home_team_0.5-,home_team_0.5+,home_team_1.5+,home_team_2.5+,home_team_3.5+,home_team_4.5+,home_clean_sheet,home_or_draw_and_over_1.5,home_win_clean_sheet,home_goals_sum_last_5,home_goals_mean_last_5,home_goals_std_last_5,home_conceded_sum_last_5,home_dominance,home_points_last_5,home_win_streak,home_cumulative_points_season,away_points,away_won,away_drew,away_lost,away_won_or_drew,away_team_0.5-,away_team_0.5+,away_team_1.5+,away_team_2.5+,away_team_3.5+,away_team_4.5+,away_clean_sheet,away_or_draw_and_over_1.5,away_win_clean_sheet,away_goals_sum_last_5,away_goals_mean_last_5,away_goals_std_last_5,away_conceded_sum_last_5,away_dominance,away_points_last_5,away_win_streak,away_cumulative_points_season,attack_vs_defense_home,attack_vs_defense_away,form_differential_last_5,h2h_key,h2h_home_win_rate_last_5,day_of_week,month,is_weekend,home_rest_days,away_rest_days,rest_advantage
5488,Brazil,Serie A,2026,2026-05-30,20:00,Flamengo RJ,Coritiba,3,0,H,1.36,5.5,9.5,1.32,5.08,8.16,1.37,5.6,10.0,1.3,5.5,8.0,3,0,1,1,1,0,0,0,0,0,0,0,3,3,3,1,0,0,1,0,1,1,1,0,0,1,1,1,10,2.0,1.224745,6,4.0,10,0,14.0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,6,1.2,1.303840,9,0.25,4,1,14.0,0.2,0.0,6,"(Coritiba, Flamengo RJ)",0.800000,5,5,1,6.0,13.0,-7.0
5489,Brazil,Serie A,2026,2026-05-30,21:30,Bahia,Botafogo RJ,2,1,H,2.0,3.8,3.6,1.95,3.63,3.43,2.1,3.75,3.8,1.96,3.75,3.5,3,0,1,1,1,0,0,0,1,1,1,0,1,1,3,1,0,0,1,0,1,1,0,0,0,0,1,0,8,1.6,0.894427,7,1.5,5,0,10.0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,9,1.8,1.303840,8,0.67,8,0,11.0,0.0,0.4,-3,"(Bahia, Botafogo RJ)",0.600000,5,5,1,13.0,7.0,6.0
5494,Brazil,Serie A,2026,2026-05-31,20:00,Vasco,Atletico-MG,0,1,A,1.96,3.6,4.1,1.9,3.4,3.87,2.02,3.65,4.2,1.81,3.5,4.1,1,0,1,0,0,0,0,0,0,0,0,0,-1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,6,1.2,0.836660,7,0.5,9,0,16.0,3,1,0,0,1,0,1,0,0,0,0,1,0,1,7,1.4,1.949359,5,2.0,6,0,6.0,0.2,0.0,3,"(Atletico-MG, Vasco)",0.800000,6,5,1,6.0,7.0,-1.0
5493,Brazil,Serie A,2026,2026-05-31,20:00,Palmeiras,Chapecoense-SC,1,0,H,1.44,4.8,9.0,1.37,4.63,7.74,1.44,5.1,8.8,1.37,4.75,7.0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,3,1,0,0,1,0,1,0,0,0,0,1,0,1,7,1.4,0.547723,4,2.0,11,0,20.0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,3,0.6,0.547723,9,0.5,1,0,2.0,-0.4,-0.2,10,"(Chapecoense-SC, Palmeiras)",0.400000,6,5,1,14.0,7.0,7.0
5491,Brazil,Serie A,2026,2026-05-31,00:00,Santos,Vitoria,3,1,H,1.74,3.9,5.75,1.64,3.67,5.11,1.71,3.95,5.9,1.57,3.7,5.75,4,0,1,1,1,1,0,0,1,1,1,1,2,2,3,1,0,0,1,0,1,1,1,0,0,0,1,0,7,1.4,0.894427,6,2.0,9,0,14.0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,4,0.8,0.836660,11,0.5,2,0,3.0,-0.8,-0.4,7,"(Santos, Vitoria)",0.200000,6,5,1,14.0,14.0,0.0
5492,Brazil,Serie A,2026,2026-05-31,15:00,Bragantino,Internacional,3,1,H,2.02,3.55,3.8,1.98,3.41,3.54,2.04,3.65,4.0,1.96,3.4,3.7,4,0,1,1,1,1,0,0,1,1,1,1,2,2,3,1,0,0,1,0,1,1,1,0,0,0,1,0,10,2.0,1.581139,5,2.0,9,1,13.0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,7,1.4,0.894427,7,0.5,8,0,10.0,0.6,0.4,1,"(Bragantino, Internacional)",0.600000,6,5,1,14.0,8.0,6.0
5495,Brazil,Serie A,2026,2026-06-01,00:30,Cruzeiro,Fluminense,1,1,D,1.91,3.4,4.6,1.87,3.26,4.21,1.97,3.55,4.6,1.86,3.4,4.33,2,0,1,1,0,0,0,0,1,1,0,0,0,0,1,0,1,0,1,0,1,0,0,0,0,0,0,0,10,2.0,0.707107,5,1.0,12,1,15.0,1,0,1,0,1,0,1,0,0,0,0,0,0,0,6,1.2,1.303840,9,1.0,4,0,8.0,0.2,0.2,8,"(Cruzeiro, Fluminense)",0.600000,0,6,0,8.0,9.0,-1.0
5496,Brazil,Serie A,2026,2026-06-01,00:30,Remo,Sao Paulo,1,0,H,3.14,3.3,2.4,2.95,3.17,2.35,3.3,3.35,2.48,3.1,3.2,2.3,1,0,1,0,0,0,0,0,0,0,0,0,1,1,3,1,0,0,1,0,1,0,0,0,0,1,0,1,7,1.4,1.516575,6,2.0,5,0,7.0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,5,1.0,0.707107,10,0.5,1,0,8.0,-0.6,-0.2,4,"(Remo, Sao Paulo)",NaN,0,6,0,8.0,16.0,-8.0
5497,Brazil,Serie A,2026,2026-07-16,23:30,Botafogo RJ,Santos,2,1,H,2.26,3.5,3.35,2.17,3.33,3.15,2.32,3.5,3.45,2.25,3.3,3.2,3,0,1,1,1,0,0,0,

### Elo ratings

In [11]:
def add_elo_ratings(
    df,
    home_col="Home",
    away_col="Away",
    home_goals_col="HG",
    away_goals_col="AG",
    date_col="Date",
    season_col="Season",
    initial_rating=1500,
    k_factor=20,
    home_advantage=100,
    use_margin_of_victory=True,
    regress_to_mean=True,
    regression_factor=0.75,  # 1.0 = no regression, 0.0 = full reset to mean
):
    """
    Adds pre-match Elo ratings for home and away teams, sequentially updated
    match by match in chronological order, with optional season-transition
    regression to the mean (accounts for squad turnover in the offseason).
    """
    df = df.sort_values(date_col).reset_index(drop=True)

    ratings = {}       # team -> current elo rating
    last_season = {}   # team -> last season they played in

    home_elo_pre = np.empty(len(df))
    away_elo_pre = np.empty(len(df))
    home_elo_post = np.empty(len(df))
    away_elo_post = np.empty(len(df))
    expected_home = np.empty(len(df))

    def get_current_rating(team, season):
        r = ratings.get(team, initial_rating)

        if regress_to_mean and season_col is not None:
            prev_season = last_season.get(team)
            if prev_season is not None and prev_season != season:
                league_mean = np.mean(list(ratings.values())) if ratings else initial_rating
                r = league_mean + (r - league_mean) * regression_factor
                ratings[team] = r

            last_season[team] = season

        return r

    for i, row in df.iterrows():
        home = row[home_col]
        away = row[away_col]
        hg = row[home_goals_col]
        ag = row[away_goals_col]
        season = row[season_col] if season_col is not None else None

        r_home = get_current_rating(home, season)
        r_away = get_current_rating(away, season)

        home_elo_pre[i] = r_home
        away_elo_pre[i] = r_away

        rating_diff = (r_home + home_advantage) - r_away
        exp_home = 1 / (1 + 10 ** (-rating_diff / 400))
        expected_home[i] = exp_home

        if pd.isna(hg) or pd.isna(ag):
            home_elo_post[i] = r_home
            away_elo_post[i] = r_away
            continue

        if hg > ag:
            actual_home = 1.0
        elif hg == ag:
            actual_home = 0.5
        else:
            actual_home = 0.0

        if use_margin_of_victory:
            goal_diff = abs(hg - ag)
            mov_multiplier = np.log(goal_diff + 1) * (2.2 / (abs(rating_diff) * 0.001 + 2.2))
            mov_multiplier = max(mov_multiplier, 1.0) if goal_diff > 0 else 1.0
        else:
            mov_multiplier = 1.0

        k_adj = k_factor * mov_multiplier

        r_home_new = r_home + k_adj * (actual_home - exp_home)
        r_away_new = r_away + k_adj * ((1 - actual_home) - (1 - exp_home))

        ratings[home] = r_home_new
        ratings[away] = r_away_new

        home_elo_post[i] = r_home_new
        away_elo_post[i] = r_away_new

    df["home_elo_pre"] = home_elo_pre
    df["away_elo_pre"] = away_elo_pre
    df["elo_diff"] = (df["home_elo_pre"] + home_advantage) - df["away_elo_pre"]
    df["elo_expected_home_win_prob"] = expected_home
    df["home_elo_post"] = home_elo_post
    df["away_elo_post"] = away_elo_post

    return df

df = add_elo_ratings(df)
df.tail(30)

,Country,League,Season,Date,Time,Home,Away,HG,AG,Res,MaxCH,MaxCD,MaxCA,AvgCH,AvgCD,AvgCA,BFECH,BFECD,BFECA,B365CH,B365CD,B365CA,TG,0.5-,0.5+,1.5+,2.5+,3.5+,4.5+,5.5+,BTTS,over_1.5+_and_btts,over_2.5+_and_btts,over_3.5+_and_btts,score_difference,absolute_score_difference,home_points,home_won,home_drew,home_lost,home_won_or_drew,home_team_0.5-,home_team_0.5+,home_team_1.5+,home_team_2.5+,home_team_3.5+,home_team_4.5+,home_clean_sheet,home_or_draw_and_over_1.5,home_win_clean_sheet,home_goals_sum_last_5,home_goals_mean_last_5,home_goals_std_last_5,home_conceded_sum_last_5,home_dominance,home_points_last_5,home_win_streak,home_cumulative_points_season,away_points,away_won,away_drew,away_lost,away_won_or_drew,away_team_0.5-,away_team_0.5+,away_team_1.5+,away_team_2.5+,away_team_3.5+,away_team_4.5+,away_clean_sheet,away_or_draw_and_over_1.5,away_win_clean_sheet,away_goals_sum_last_5,away_goals_mean_last_5,away_goals_std_last_5,away_conceded_sum_last_5,away_dominance,away_points_last_5,away_win_streak,away_cumulative_points_season,attack_vs_defense_home,attack_vs_defense_away,form_differential_last_5,h2h_key,h2h_home_win_rate_last_5,day_of_week,month,is_weekend,home_rest_days,away_rest_days,rest_advantage,home_elo_pre,away_elo_pre,elo_diff,elo_expected_home_win_prob,home_elo_post,away_elo_post
5488,Brazil,Serie A,2026,2026-05-30,20:00,Athletico-PR,Mirassol,1,0,H,1.95,3.33,4.75,1.88,3.21,4.28,2.0,3.4,4.7,1.86,3.3,4.33,1,0,1,0,0,0,0,0,0,0,0,0,1,1,3,1,0,0,1,0,1,0,0,0,0,1,0,1,10,2.0,1.581139,3,2.0,11,0,20.0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,5,1.0,1.000000,9,0.5,3,0,4.0,0.2,0.4,8,"(Athletico-PR, Mirassol)",NaN,5,5,1,13.0,14.0,-1.0,1457.749322,1470.608230,87.141092,0.622841,1465.292502,1463.065050
5489,Brazil,Serie A,2026,2026-05-30,20:00,Flamengo RJ,Coritiba,3,0,H,1.36,5.5,9.5,1.32,5.08,8.16,1.37,5.6,10.0,1.3,5.5,8.0,3,0,1,1,1,0,0,0,0,0,0,0,3,3,3,1,0,0,1,0,1,1,1,0,0,1,1,1,10,2.0,1.224745,6,4.0,10,0,14.0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,6,1.2,1.303840,9,0.25,4,1,14.0,0.2,0.0,6,"(Coritiba, Flamengo RJ)",0.800000,5,5,1,6.0,13.0,-7.0,1597.710498,1432.694029,265.016469,0.821355,1602.131068,1428.273459
5490,Brazil,Serie A,2026,2026-05-31,15:00,Bragantino,Internacional,3,1,H,2.02,3.55,3.8,1.98,3.41,3.54,2.04,3.65,4.0,1.96,3.4,3.7,4,0,1,1,1,1,0,0,1,1,1,1,2,2,3,1,0,0,1,0,1,1,1,0,0,0,1,0,10,2.0,1.581139,5,2.0,9,1,13.0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,7,1.4,0.894427,7,0.5,8,0,10.0,0.6,0.4,1,"(Bragantino, Internacional)",0.600000,6,5,1,14.0,8.0,6.0,1466.234170,1450.510107,115.724063,0.660643,1473.317982,1443.426294
5491,Brazil,Serie A,2026,2026-05-31,00:00,Santos,Vitoria,3,1,H,1.74,3.9,5.75,1.64,3.67,5.11,1.71,3.95,5.9,1.57,3.7,5.75,4,0,1,1,1,1,0,0,1,1,1,1,2,2,3,1,0,0,1,0,1,1,1,0,0,0,1,0,7,1.4,0.894427,6,2.0,9,0,14.0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,4,0.8,0.836660,11,0.5,2,0,3.0,-0.8,-0.4,7,"(Santos, Vitoria)",0.200000,6,5,1,14.0,14.0,0.0,1429.112567,1455.795165,73.317403,0.603973,1437.533523,1447.374209
5492,Brazil,Serie A,2026,2026-05-31,20:00,Vasco,Atletico-MG,0,1,A,1.96,3.6,4.1,1.9,3.4,3.87,2.02,3.65,4.2,1.81,3.5,4.1,1,0,1,0,0,0,0,0,0,0,0,0,-1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,6,1.2,0.836660,7,0.5,9,0,16.0,3,1,0,0,1,0,1,0,0,0,0,1,0,1,7,1.4,1.949359,5,2.0,6,0,6.0,0.2,0.0,3,"(Atletico-MG, Vasco)",0.800000,6,5,1,6.0,7.0,-1.0,1426.736416,1448.742081,77.994334,0.610395,1414.528522,1460.949975
5493,Brazil,Serie A,2026,2026-05-31,20:00,Palmeiras,Chapecoense-SC,1,0,H,1.44,4.8,9.0,1.37,4.63,7.74,1.44,5.1,8.8,1.37,4.75,7.0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,3,1,0,0,1,0,1,0,0,0,0,1,0,1,7,1.4,0.547723,4,2.0,11,0,20.0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,3,0.6,0.547723,9,0.5,1,0,2.0,-0.4,-0.2,10,"(Chapecoense-SC, Palmeiras)",0.400000,6,5,1,14.0,7.0,7.0,1622.067290,1297.223731,424.843559,0.920239,1623.662514,1295.628507
5494,Brazil,Serie A,2026,2026-06-01,00:30,Cruzeiro,Fluminense,1,1,D,1.91,3.4,4.6,1.87,3.26,4.21,1.97,3.55,4.6,1.86,3.4,4.33,2,0,1,1,0,0,0,0,1,1,0,0,0,0,1,0,1,0,1,0,1,0,0,0,0,0,0,0,10,2.0,0.707107,5,1.0,12,1,15.0,1,0,1,0,1,0,1,0,0,0,0,0,0,0,6,1.2,1.303840,9,1.0,4,0,8.0,0.2,0.2,8,"(Cruze

### Glicko ratings


In [12]:
def add_glicko_ratings(
    df,
    home_col="Home",
    away_col="Away",
    home_goals_col="HG",
    away_goals_col="AG",
    date_col="Date",
    initial_rating=1500,
    initial_rd=350,
    rd_min=50,
    rd_max=350,
    home_advantage=100,
    rd_recovery_days=180,  # days of inactivity for RD to fully return to rd_max
):
    """
    Adds pre-match Glicko-1 ratings + rating deviations (RD) for home/away teams,
    sequentially updated match by match (each match = its own rating period).
    """
    df = df.sort_values(date_col).reset_index(drop=True)

    Q = math.log(10) / 400
    C = math.sqrt((rd_max**2 - rd_min**2) / rd_recovery_days) if rd_recovery_days > 0 else 0

    ratings = {}
    rds = {}
    last_played = {}

    n = len(df)
    home_rating_pre = np.empty(n)
    away_rating_pre = np.empty(n)
    home_rd_pre = np.empty(n)
    away_rd_pre = np.empty(n)
    home_rating_post = np.empty(n)
    away_rating_post = np.empty(n)
    home_rd_post = np.empty(n)
    away_rd_post = np.empty(n)
    expected_home = np.empty(n)

    def g(rd):
        return 1 / math.sqrt(1 + 3 * Q**2 * rd**2 / math.pi**2)

    def expected_score(r, r_opp, rd_opp):
        return 1 / (1 + 10 ** (-g(rd_opp) * (r - r_opp) / 400))

    def decayed_rd(team, current_date):
        r = ratings.get(team, initial_rating)
        rd = rds.get(team, initial_rd)
        last = last_played.get(team)
        if last is not None:
            days_idle = (current_date - last).days
            rd = min(rd_max, math.sqrt(rd**2 + (C**2) * days_idle))
        return r, rd

    def glicko_update(r, rd, r_opp, rd_opp, score):
        g_opp = g(rd_opp)
        E = expected_score(r, r_opp, rd_opp)
        d2 = 1 / (Q**2 * g_opp**2 * E * (1 - E)) if E not in (0, 1) else float("inf")
        new_rd = math.sqrt(1 / (1 / rd**2 + 1 / d2)) if d2 != float("inf") else rd
        new_rd = max(rd_min, min(rd_max, new_rd))
        new_r = r + Q / (1 / rd**2 + 1 / d2) * g_opp * (score - E) if d2 != float("inf") else r
        return new_r, new_rd, E

    for i, row in df.iterrows():
        home = row[home_col]
        away = row[away_col]
        hg = row[home_goals_col]
        ag = row[away_goals_col]
        current_date = row[date_col]

        r_home, rd_home = decayed_rd(home, current_date)
        r_away, rd_away = decayed_rd(away, current_date)

        home_rating_pre[i] = r_home
        away_rating_pre[i] = r_away
        home_rd_pre[i] = rd_home
        away_rd_pre[i] = rd_away

        exp_home = expected_score(r_home + home_advantage, r_away, rd_away)
        expected_home[i] = exp_home

        if pd.isna(hg) or pd.isna(ag):
            home_rating_post[i] = r_home
            away_rating_post[i] = r_away
            home_rd_post[i] = rd_home
            away_rd_post[i] = rd_away
            continue

        if hg > ag:
            score_home, score_away = 1.0, 0.0
        elif hg == ag:
            score_home, score_away = 0.5, 0.5
        else:
            score_home, score_away = 0.0, 1.0

        new_r_home, new_rd_home, _ = glicko_update(r_home, rd_home, r_away, rd_away, score_home)
        new_r_away, new_rd_away, _ = glicko_update(r_away, rd_away, r_home, rd_home, score_away)

        ratings[home] = new_r_home
        ratings[away] = new_r_away
        rds[home] = new_rd_home
        rds[away] = new_rd_away
        last_played[home] = current_date
        last_played[away] = current_date

        home_rating_post[i] = new_r_home
        away_rating_post[i] = new_r_away
        home_rd_post[i] = new_rd_home
        away_rd_post[i] = new_rd_away

    df["home_rating_pre"] = home_rating_pre
    df["away_rating_pre"] = away_rating_pre
    df["home_rd_pre"] = home_rd_pre
    df["away_rd_pre"] = away_rd_pre
    df["glicko_expected_home_win_prob"] = expected_home
    df["home_rating_post"] = home_rating_post
    df["away_rating_post"] = away_rating_post
    df["home_rd_post"] = home_rd_post
    df["away_rd_post"] = away_rd_post

    return df

df = add_glicko_ratings(df)
df.tail(30)

,Country,League,Season,Date,Time,Home,Away,HG,AG,Res,MaxCH,MaxCD,MaxCA,AvgCH,AvgCD,AvgCA,BFECH,BFECD,BFECA,B365CH,B365CD,B365CA,TG,0.5-,0.5+,1.5+,2.5+,3.5+,4.5+,5.5+,BTTS,over_1.5+_and_btts,over_2.5+_and_btts,over_3.5+_and_btts,score_difference,absolute_score_difference,home_points,home_won,home_drew,home_lost,home_won_or_drew,home_team_0.5-,home_team_0.5+,home_team_1.5+,home_team_2.5+,home_team_3.5+,home_team_4.5+,home_clean_sheet,home_or_draw_and_over_1.5,home_win_clean_sheet,home_goals_sum_last_5,home_goals_mean_last_5,home_goals_std_last_5,home_conceded_sum_last_5,home_dominance,home_points_last_5,home_win_streak,home_cumulative_points_season,away_points,away_won,away_drew,away_lost,away_won_or_drew,away_team_0.5-,away_team_0.5+,away_team_1.5+,away_team_2.5+,away_team_3.5+,away_team_4.5+,away_clean_sheet,away_or_draw_and_over_1.5,away_win_clean_sheet,away_goals_sum_last_5,away_goals_mean_last_5,away_goals_std_last_5,away_conceded_sum_last_5,away_dominance,away_points_last_5,away_win_streak,away_cumulative_points_season,attack_vs_defense_home,attack_vs_defense_away,form_differential_last_5,h2h_key,h2h_home_win_rate_last_5,day_of_week,month,is_weekend,home_rest_days,away_rest_days,rest_advantage,home_elo_pre,away_elo_pre,elo_diff,elo_expected_home_win_prob,home_elo_post,away_elo_post,home_rating_pre,away_rating_pre,home_rd_pre,away_rd_pre,glicko_expected_home_win_prob,home_rating_post,away_rating_post,home_rd_post,away_rd_post
5488,Brazil,Serie A,2026,2026-05-30,21:30,Gremio,Corinthians,1,3,A,3.15,3.2,2.5,2.99,3.08,2.38,3.3,3.15,2.58,3.0,3.2,2.4,4,0,1,1,1,1,0,0,1,1,1,1,-2,2,0,0,0,1,0,0,1,0,0,0,0,0,0,0,6,1.2,1.303840,3,0.5,10,1,17.0,3,1,0,0,1,0,1,1,1,0,0,0,1,0,3,0.6,0.547723,8,2.0,2,0,7.0,-0.4,0.0,8,"(Corinthians, Gremio)",0.200000,5,5,1,7.0,13.0,-6.0,1458.692094,1455.816219,102.875874,0.643870,1445.176821,1469.331492,1649.180961,1621.866715,173.910471,174.979614,0.654915,1580.873638,1690.953864,159.383554,160.168197
5489,Brazil,Serie A,2026,2026-05-30,20:00,Athletico-PR,Mirassol,1,0,H,1.95,3.33,4.75,1.88,3.21,4.28,2.0,3.4,4.7,1.86,3.3,4.33,1,0,1,0,0,0,0,0,0,0,0,0,1,1,3,1,0,0,1,0,1,0,0,0,0,1,0,1,10,2.0,1.581139,3,2.0,11,0,20.0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,5,1.0,1.000000,9,0.5,3,0,4.0,0.2,0.4,8,"(Athletico-PR, Mirassol)",NaN,5,5,1,13.0,14.0,-1.0,1457.749322,1470.608230,87.141092,0.622841,1465.292502,1463.065050,1692.348679,1537.386408,175.164347,183.358679,0.780491,1733.817677,1492.449167,162.339614,168.549430
5490,Brazil,Serie A,2026,2026-05-31,20:00,Palmeiras,Chapecoense-SC,1,0,H,1.44,4.8,9.0,1.37,4.63,7.74,1.44,5.1,8.8,1.37,4.75,7.0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,3,1,0,0,1,0,1,0,0,0,0,1,0,1,7,1.4,0.547723,4,2.0,11,0,20.0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,3,0.6,0.547723,9,0.5,1,0,2.0,-0.4,-0.2,10,"(Chapecoense-SC, Palmeiras)",0.400000,6,5,1,14.0,7.0,7.0,1622.067290,1297.223731,424.843559,0.920239,1623.662514,1295.628507,1873.783060,1268.575773,187.248971,192.563714,0.969642,1881.849997,1260.177160,183.685930,188.714345
5491,Brazil,Serie A,2026,2026-05-31,20:00,Vasco,Atletico-MG,0,1,A,1.96,3.6,4.1,1.9,3.4,3.87,2.02,3.65,4.2,1.81,3.5,4.1,1,0,1,0,0,0,0,0,0,0,0,0,-1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,6,1.2,0.836660,7,0.5,9,0,16.0,3,1,0,0,1,0,1,0,0,0,0,1,0,1,7,1.4,1.949359,5,2.0,6,0,6.0,0.2,0.0,3,"(Atletico-MG, Vasco)",0.800000,6,5,1,6.0,7.0,-1.0,1426.736416,1448.742081,77.994334,0.610395,1414.528522,1460.949975,1557.925736,1559.211385,175.692665,177.738816,0.621268,1493.324486,1625.196819,160.788047,162.277262
5492,Brazil,Serie A,2026,2026-05-31,15:00,Bragantino,Internacional,3,1,H,2.02,3.55,3.8,1.98,3.41,3.54,2.04,3.65,4.0,1.96,3.4,3.7,4,0,1,1,1,1,0,0,1,1,1,1,2,2,3,1,0,0,1,0,1,1,1,0,0,0,1,0,10,2.0,1.581139,5,2.0,9,1,13.0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,7,1.4,0.894427,7,0.5,8,0,10.0,0.6,0.4,1,"(Bragantino, Internacional)",0.600000,6,5,1,14.0,8.0,6.0,1466.234170,1450.510107,115.724063,0.660643,1473.317982,1443.426294,1684.124382,1614.067389,176.621011,180.026144,0.700562,1738.328945,1557.975746,161.978013,164.470689
5493,Brazil,Serie A,2026,2026-05